# What Happens When a DataFrame Action Is Called?

## Spark 3.5+ job, stage, and task internals

From a lazy DataFrame plan to distributed execution, retries, and a returned result.

# Learning goals

By the end, you should be able to:

- trace an action through Catalyst, `QueryExecution`, `SparkContext`, and the schedulers;
- explain why a job has one or more stages and why each stage has many tasks;
- distinguish narrow dependencies from shuffle (wide) dependencies;
- explain branches, skipped stages, stage completion, and Adaptive Query Execution (AQE);
- reason about task retry, fetch failure, stage retry, and job failure; and
- investigate the Spark UI and event logs instead of guessing.

# Vocabulary: do not mix these levels

| Unit | Meaning | Usually created by | Parallelism |
|---|---|---|---|
| Query | One DataFrame/SQL computation | Catalyst/SQL execution | May cause multiple jobs |
| Job | Work submitted by an action (or internal materialization) | `DAGScheduler` receives a job submission | Its ready stages can run concurrently |
| Stage | Maximal set of pipelined work with no intervening shuffle boundary | `DAGScheduler` | Independent ready stages may overlap |
| Task | One stage's work for one partition | `TaskSchedulerImpl` | Runs in an executor slot |
| Attempt | A try at running a task or stage | Scheduler retry logic | Speculative duplicates may coexist |

> Useful model: **action → job(s) → stage(s) → task attempts**. It is not always exactly one action to one job.

# Before the action: only a recipe exists

```python
result = (orders
          .filter("status = 'COMPLETE'")
          .groupBy("country")
          .sum("amount"))
```

Transformations construct a logical plan on the driver. They normally do not scan all input or launch executor tasks.

```text
DataFrame object
    └── unresolved/resolved logical plan
          └── Filter → Aggregate → Project

No action yet → no Spark job → no stages → no tasks
```

# The end-to-end path

![DataFrame action execution path](assets/action_execution_path.svg)

ASCII fallback: `action → QueryExecution/Catalyst → SparkPlan → RDD lineage → DAGScheduler → TaskScheduler → executors`.

## Components and responsibilities

- **Catalyst / `QueryExecution`**: resolves names and types, rewrites the logical plan, chooses and prepares a physical plan.
- **SparkPlan operators**: executable SQL operators; whole-stage code generation may fuse compatible operators.
- **RDD layer**: physical operators ultimately expose partitioned computations and dependencies.
- **`DAGScheduler`**: discovers shuffle boundaries, creates stages, submits ready stages, and handles stage-level recovery.
- **`TaskSchedulerImpl`**: creates task sets, chooses executor offers with locality, tracks task attempts, retries ordinary task failures.
- **Scheduler backend / cluster manager**: supplies executor resources; the backend launches serialized tasks.
- **Executors**: run task attempts, read input/shuffle, write shuffle output, cache blocks, and return results/status.

# Catalyst's planning pipeline

![Catalyst planning pipeline](assets/catalyst_pipeline.svg)

1. **Analysis** resolves relations, attributes, functions, and compatible types.
2. **Logical optimization** applies rules such as predicate pushdown, column pruning, constant folding, and projection collapse.
3. **Physical planning** selects algorithms such as broadcast hash join, sort-merge join, and hash aggregation.
4. **Preparation** inserts exchanges and sorts needed to satisfy distribution/ordering requirements.
5. **AQE** can re-optimize at materialized shuffle boundaries using runtime statistics.

# What an action actually does

Examples include `count`, `collect`, `take`, `show`, `write`, `foreach`, and `toLocalIterator`. Their result contracts differ.

- `collect()` must compute every required output partition and transfer all rows to the driver.
- `count()` computes an aggregate; only a small final result returns, but input may still be fully scanned.
- `take(n)` / `show(n)` may probe only enough partitions to obtain rows and can submit multiple jobs while expanding the partition scan.
- `write` computes partitions and commits files through the data source's commit protocol.
- An action over cached data may launch work only for missing cache partitions.

Therefore: **an action is an execution request, not a promise of exactly one job**.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("JobInternals")
         .master("local[4]")  # remove on a managed cluster
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.version

# Narrow vs wide transformation

A dependency is **narrow** when each child partition reads from a small, fixed number of parent partitions (typically one). Work can be pipelined in one task and failure recovery can recompute a limited lineage. Examples: `map`, `filter`, `withColumn`, and many `select` operations.

A dependency is **wide** when a child partition can require records from many parent partitions. Data must be redistributed through a **shuffle**. Examples: `groupBy`, `distinct`, `repartition`, order-by/range exchange, and non-broadcast joins.

```text
NARROW (same stage)                 WIDE (stage boundary)
P0 → filter → map → out0           P0 ─┐              ┌→ reduce partition 0
P1 → filter → map → out1           P1 ─┼→ shuffle ────┤
P2 → filter → map → out2           P2 ─┘              └→ reduce partition 1
```

> `coalesce(n)` without shuffle is usually narrow; `repartition(n)` deliberately shuffles. Operator names alone are not enough—inspect the physical plan for `Exchange`.

# Exchange is the SQL-level boundary marker

In a physical DataFrame plan, an `Exchange` establishes a required distribution, commonly hash partitioning, range partitioning, single partition, or broadcast distribution. A non-broadcast shuffle exchange normally separates shuffle-map work from downstream result work.

```text
Scan → Filter → PartialHashAggregate
                 │
                 ▼
       Exchange hashpartitioning(key, 8)
                 │
                 ▼
          FinalHashAggregate → Result

       Stage 0 writes shuffle │ Stage 1 fetches shuffle
```

`BroadcastExchange` is materialized separately and can create an additional internal job/query stage under SQL execution.

In [ ]:
events = (spark.range(0, 1_000_000, 1, 8)
          .select((F.col("id") % 100).alias("customer_id"),
                  (F.col("id") % 7).alias("day"),
                  (F.col("id") * 0.01).alias("amount")))

summary = (events
           .filter(F.col("amount") > 10)        # narrow
           .withColumn("tax", F.col("amount") * 0.18)  # narrow
           .groupBy("day")                     # wide: shuffle required
           .agg(F.sum("tax").alias("tax_total")))

summary.explain(mode="formatted")

# Exactly how stages are created

When a terminal RDD operation calls `SparkContext.runJob`, the `DAGScheduler` walks backward from the requested final RDD partitions.

```text
1. Create a ResultStage for the requested output partitions.
2. Walk its RDD dependencies backward.
3. Pipeline narrow dependencies into that stage.
4. At every unresolved ShuffleDependency:
      create/find a ShuffleMapStage for its map side,
      then recursively discover its parents.
5. Submit parent stages with no missing parents.
6. When parents finish, submit newly unblocked children.
```

A **ShuffleMapStage** writes partitioned shuffle blocks. A **ResultStage** runs the action's final function and returns results (or performs output writes). Stage IDs are application-wide identifiers and need not be contiguous within one job.

# One job, branching stage DAG

Consider a sort-merge join whose two inputs each need repartitioning:

![Branching stage DAG](assets/branching_stage_dag.svg)

Stages 0 and 1 have no dependency on each other. If executor resources are available, the scheduler may run tasks from both at the same time. Stage 2 waits for all required map outputs from both parent shuffles.

If a later aggregate adds another exchange, the join stage may itself become a shuffle-map stage and a new result stage follows. A job is a DAG, not necessarily a straight line.

In [ ]:
customers = spark.range(100).select(
    F.col("id").alias("customer_id"),
    (F.col("id") % 5).alias("segment"))

# Disable automatic broadcast only for this demonstration, then restore it.
previous_threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
try:
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
    joined = events.join(customers, "customer_id").groupBy("segment").count()
    joined.explain(mode="formatted")
finally:
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", previous_threshold)

# Look for two input Exchange nodes before the join and another before aggregation.

# Tasks inside each stage

A stage has one task per partition that the stage must process. All tasks in a stage run the same serialized computation against different partition IDs.

- A shuffle-map stage commonly has one task per upstream input partition.
- A downstream stage commonly has one task per shuffle partition after any AQE coalescing.
- A result stage can request a subset of partitions (`take` is an important example).
- Executor cores provide task slots; 1,000 tasks do not imply 1,000 simultaneous tasks.
- Preferred locations allow process-, node-, or rack-local scheduling when possible.

Each task attempt deserializes its closure, initializes metrics, reads its partition, runs the operator pipeline, writes shuffle/result output, and reports completion or failure.

# Why does the UI show a skipped stage?

A skipped stage is generally a stage whose output is already available and valid, so Spark does not rerun its tasks for this job. Typical reasons:

- shuffle-map outputs created by an earlier job are still registered and fetchable;
- a reused exchange/query stage supplies already materialized output;
- cached partitions satisfy part or all of the lineage (the physical plan often changes to `InMemoryTableScan`); or
- AQE reuses a completed query stage while producing the final adaptive plan.

Skipped does **not** mean Catalyst proved the computation mathematically unnecessary. It usually means the needed physical output already exists. If an executor holding shuffle files is lost and external shuffle preservation is unavailable, those outputs may become missing and Spark must recompute the map stage.

In [ ]:
# Observe reuse/cache behavior in the Spark UI.
spark.sparkContext.setJobGroup("cache-build", "Materialize cached aggregation")
cached = summary.cache()
cached.count()

spark.sparkContext.setJobGroup("cache-reuse", "Read cached aggregation")
cached.orderBy("day").collect()

# Cleanup when finished with this experiment.
cached.unpersist()

# What happens when a task ends?

![Successful task completion sequence](assets/task_completion.svg)

For shuffle-map tasks, the executor commits shuffle files and reports `MapStatus` describing block locations and sizes. For result tasks, the result is delivered to the driver's result handler (large results may be fetched indirectly through the block manager). Failed, killed, or superseded attempts do not count as successful partition completion.

# What happens when a stage ends?

A stage is successful when every required output partition has a successful task attempt. Then the driver:

1. removes the stage from the running set and records completion;
2. retains/records shuffle output locations for a shuffle-map stage;
3. emits listener events used by the UI and event log;
4. checks waiting child stages and submits any whose parents are now available;
5. for an AQE query stage, exposes runtime statistics and may trigger re-optimization; and
6. if this was the final result stage, completes the job and lets the action return or the write commit proceed.

Stage completion is a driver-side scheduling fact. It does not mean shuffle files were copied to the driver; they remain distributed for downstream fetches.

# AQE changes the plan during execution

Spark 3.5 commonly runs SQL with Adaptive Query Execution enabled. Exchanges divide the physical plan into **query stages** that can be materialized. Runtime shuffle statistics then allow AQE to:

- coalesce small shuffle partitions;
- split/handle skewed partitions;
- convert a sort-merge join to a broadcast or shuffled-hash join when appropriate; and
- optimize local shuffle reads.

```text
Initial SparkPlan → materialize query stage → collect map statistics
                                      ↓
                          rewrite remaining plan
                                      ↓
                           final adaptive plan
```

A SQL **query stage** is an AQE planning/materialization unit; a scheduler **stage** is a `DAGScheduler` execution unit. They are related but not identical concepts.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

adaptive_demo = events.groupBy("day").agg(F.avg("amount").alias("avg_amount"))
adaptive_demo.collect()  # materializes the adaptive plan
adaptive_demo.explain(mode="formatted")

# In the executed plan, compare the Final Plan with the Initial Plan.

# Failure hierarchy

![Spark failure-handling hierarchy](assets/failure_hierarchy.svg)

The earlier Mermaid version used short internal aliases: `K` meant failure-kind decision, `RT` task retry, `OK` success decision, `MS` missing shuffle, `RR` repeated failure, `CONT` continue, `RF` recompute/retry, and `AB` abort. These abbreviations are not Spark classes, states, or configuration names; the SVG now displays the full meanings directly.

A rectangle `[like this]` represents an action/state. A diamond-shaped node `{like this}` represents a decision. Arrow labels such as `|FetchFailed|` state which condition selects that path.

Recovery occurs at the smallest safe unit: retry an attempt first; recompute a lost shuffle dependency when required; fail the stage/job when retry policy or an unrecoverable condition is reached.

# Ordinary task failure

If one task attempt throws, exits, is killed with its executor, or is lost:

- `TaskSchedulerImpl` records the failure and normally schedules another attempt, preferably on a healthy executor;
- other successful partitions are not recomputed just because one attempt failed;
- the default `spark.task.maxFailures` is normally 4 continuous failures for a particular task before the job is aborted (cluster managers/deployments can alter behavior);
- speculation is different: it launches a duplicate of a slow task, and the first successful attempt wins; and
- executor loss can invalidate cached blocks and shuffle outputs located on that executor.

A deterministic exception—bad parsing logic, a Python UDF bug, divide-by-zero under ANSI mode—usually fails again on the same record. Retrying cannot repair code or data.

# Fetch failure and stage retry

A reduce-side task fetches shuffle blocks from every required map output. `FetchFailed` means a required block cannot be obtained or validated—for example, its executor/file disappeared or transfer repeatedly failed.

```text
Stage A attempt 0: map outputs M0 M1 M2 M3
                              X  (M2 lost)
Stage B attempt 0: fetch → FetchFailed → stage attempt fails
                              │
                              ├→ recompute missing M2 in Stage A attempt 1
                              └→ retry Stage B as a new stage attempt
```

Spark invalidates the missing map output and resubmits necessary producer work before retrying the consumer. Repeated stage attempts beyond configured limits cause job abortion. This is why a completed shuffle stage can later appear again.

# Stage failure vs job failure

A **task attempt failure** is local. A **stage attempt failure** means the current attempt cannot complete, but the scheduler may retry the stage. A **job failure** is terminal for that action.

Common terminal causes include:

- the same task exceeds its allowed failure count;
- repeated fetch/stage failures exceed scheduler limits;
- serialization, analysis, or driver-side planning errors;
- driver result size limits or driver/executor out-of-memory conditions;
- explicit cancellation, context shutdown, or cluster loss; and
- output commit failures that cannot be safely recovered.

One failed job does not necessarily stop the `SparkContext`; independent jobs may continue unless the application or shared resource failure prevents it.

# Side effects and retries: an important trap

Spark provides retryable computation, not exactly-once execution of arbitrary side effects. A task attempt may run more than once because of failure or speculation.

Avoid unguarded external side effects inside `map`, UDFs, or `foreachPartition`, such as:

- charging a payment;
- incrementing a remote counter;
- sending an email; or
- inserting without an idempotency key.

Prefer idempotent/upsert semantics, transactional sinks, unique operation keys, and supported data-source commit protocols. Accumulators can also reflect attempt semantics and should not be treated as an exact business ledger.

# Safe failure experiment

The next cell intentionally fails an action. It demonstrates a deterministic task error. Run it only in a disposable teaching session.

Watch the Jobs/Stages tabs and executor logs. Notice repeated task attempts and the eventual action exception at the driver. The exact number and presentation can differ by master and configuration.

In [ ]:
# INTENTIONAL FAILURE — uncomment to run.
# from pyspark.sql.types import LongType
# @F.udf(LongType())
# def fail_on_42(x):
#     if x == 42:
#         raise RuntimeError("intentional deterministic failure for this lesson")
#     return x
#
# spark.sparkContext.setJobGroup("failure-demo", "Intentional task retry demo")
# spark.range(0, 100, 1, 4).select(fail_on_42("id")).collect()

# Reading `explain` without confusing plans and stages

`df.explain("extended")` shows parsed, analyzed, optimized logical, and physical plans. `df.explain("formatted")` emphasizes the physical operator tree.

Use this workflow:

1. Find scans and verify pushed filters/read schema.
2. Find `Exchange` and `BroadcastExchange` nodes.
3. Identify join strategies, aggregate phases, sorts, and partition counts.
4. Run the action.
5. Reinspect the final adaptive plan when AQE is enabled.
6. Correlate SQL operators to jobs/stages in the Spark UI.

> The physical plan predicts boundaries; the Spark UI records what actually ran. Operator count is not stage count, and exchange count is not always job count.

In [ ]:
def inspect(df, label="inspect"):
    print(f"Partitions reported before action: {df.rdd.getNumPartitions()}")
    print("\n=== EXTENDED PLAN ===")
    df.explain(mode="extended")
    spark.sparkContext.setJobGroup(label, f"Inspect {label}")
    rows = df.collect()
    print(f"\nReturned {len(rows)} rows")
    print("\n=== FORMATTED EXECUTED/ADAPTIVE PLAN ===")
    df.explain(mode="formatted")
    return rows

inspect(summary, "summary")

# Spark UI investigation checklist

Open the Spark UI (commonly linked by your notebook environment; local default is often `http://localhost:4040`).

- **SQL tab**: physical operators, duration, rows, spill, shuffle metrics, and adaptive plan.
- **Jobs tab**: action/job group, active/skipped/failed stages, and overall DAG.
- **Stages tab**: task distribution, locality, input/output, shuffle read/write, spill, GC, skew, retries.
- **Executors tab**: cores, memory/storage, task failures, lost executors, and shuffle activity.
- **Environment tab**: confirm the effective configuration rather than assuming defaults.

For completed applications, enable event logging and use the Spark History Server. Use a job group/description to make notebook actions easy to find.

In [ ]:
keys = [
    "spark.master",
    "spark.task.maxFailures",
    "spark.stage.maxConsecutiveAttempts",
    "spark.speculation",
    "spark.sql.adaptive.enabled",
    "spark.sql.shuffle.partitions",
]

for key in keys:
    try:
        print(f"{key} = {spark.conf.get(key)}")
    except Exception:
        print(f"{key} = <not explicitly available through SQLConf>")

# Misconceptions to retire

| Misconception | Better model |
|---|---|
| One action always creates one job | Some actions/internal exchanges can produce multiple jobs |
| One transformation equals one stage | Narrow operator pipelines are fused within stages |
| Every wide API call means exactly one stage | Physical exchanges and runtime reuse/AQE determine boundaries |
| Stages in a job always run sequentially | Independent parent branches can run concurrently |
| A skipped stage did no work ever | Its usable output was created earlier and reused |
| Completed stage data is stored on the driver | Shuffle/cache blocks remain distributed |
| Four task failures means four total job failures | The threshold applies to continuous failures of a particular task; details are configurable |
| Retried computations are exactly-once side effects | Task attempts can execute more than once |

# Final mental model

```text
LAZY DEFINITION (driver)
DataFrame transformations → logical plan
                              │ action
                              ▼
PLANNING (driver)
analysis → optimization → physical planning → prepared/adaptive SparkPlan
                              │ execute
                              ▼
SCHEDULING (driver)
RDD dependencies → DAG → stages at shuffle boundaries → task sets
                              │ launch
                              ▼
EXECUTION (executors)
one task attempt per required partition → shuffle blocks/results/metrics
                              │ completion or failure
                              ▼
RECOVERY/PROGRESS (driver)
retry task → recompute lost shuffle → retry stage → unblock children → finish job
```

# Lab challenges

1. Change `spark.sql.shuffle.partitions` from 8 to 3. Predict task counts, then verify.
2. Compare `repartition(8)` and `coalesce(2)` plans. Find the `Exchange`.
3. Enable and disable broadcast joins. Compare plan shape, jobs, and stages.
4. Run the cached query twice. Identify reused data and skipped work in the UI.
5. Create skew (most rows share one key), then inspect task duration and AQE skew handling.
6. Run the intentional failure demo and distinguish task attempts from stage attempts.
7. Explain why `show(20)` may not scan the same partitions or create the same jobs as `count()`.

For every experiment, write a prediction first: **jobs, stage DAG, task count per stage, shuffle bytes, and retry behavior**.